<a href="https://colab.research.google.com/github/manujeevanprakash/agentic-kyc-workflow-prototype/blob/main/KYC_Agentic_AI_Workflow_Public.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic KYC Workflow Prototype

This notebook demonstrates an agentic KYC workflow for high-net-worth client onboarding.

This prototype reflects the architecture of how an agentic KYC workflow would be implemented in a real bank, but it does not call actual APIs or databases. Instead, it uses simulated data to demonstrate the workflow structure.

The prototype uses the profile of a Canadian retired business owner who recently sold a company and is opening a new bank account.

For the full architecture walkthrough on how this would be implemented in a real bank, read the companion post:

[How to Reduce High-Net-Worth Client Onboarding Time With an Agentic KYC Workflow](https://pmexaminer.com/reduce-high-net-worth-client-onboarding-time-with-an-agentic-kyc-workflow/)

## Architecture Overview

Here's how each agent within the architecture works:

<div align="center">
  <img src="https://pmexaminer.com/wp-content/uploads/2026/05/agentic_ai_workflow_for_kyc_onboarding.webp" width="600">
</div>

Each code cell above shows one part of this workflow:
- How the case package enters the workflow
- How the Orchestrator decides which agents to run
- How specialist agents produce findings
- How the Signal Aggregator creates a unified view
- How the Risk Engine applies rules
- How an AI summary helps the compliance officer

The prototype uses Python dictionaries to represent client data, agent outputs, and workflow results. In a real bank, these would connect to CRM systems, document storage, screening vendors, and case management tools.

# Case Package

The case package is the structured client information captured during onboarding.

In a real bank, this information would come from a CRM system, onboarding forms, document management systems, and other internal banking tools.

In this prototype, I use a simple Gradio UI to demonstrate how this information could be captured.

When you click **"Generate Sample Case"**, it creates a case package for a retired business owner who:
- Recently sold a software company
- Has crypto holdings
- Expects cross-border activity
- May have PEP exposure

That case package is then passed into the KYC workflow.


In [ ]:
# @title
# Install the lightweight libraries used for the prototype UI and sample PDF generation.
!pip install -q gradio faker reportlab

from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path
from uuid import uuid4
import tempfile
import json

import gradio as gr
from faker import Faker
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas


fake = Faker()

# Clear sample-document label so nobody mistakes generated PDFs for real client evidence.
SAMPLE_NOTICE = "SAMPLE DOCUMENT - NOT REAL. FOR PROTOTYPE TESTING ONLY."

# Small set of Canadian addresses used to generate repeatable demo cases.
CANADIAN_SAMPLE_ADDRESSES = [
    "181 Bay Street, Toronto, ON",
    "1055 West Georgia Street, Vancouver, BC",
    "1250 Rene-Levesque Boulevard West, Montreal, QC",
]


def _toronto_now_iso():
    # Use Toronto time for timestamps so outputs align with the prototype's Canada focus.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


def _create_sample_pdf(file_path, title, lines):
    # Creates a simple placeholder PDF for prototype document-upload flows.
    # The file contains only basic sample text and is not used for real verification.
    c = canvas.Canvas(str(file_path), pagesize=letter)
    y = 750

    c.setFont("Helvetica-Bold", 12)
    c.drawString(72, y, SAMPLE_NOTICE)
    y -= 40

    c.drawString(72, y, title)
    y -= 30

    for line in lines:
        c.setFont("Helvetica", 10)
        c.drawString(72, y, str(line))
        y -= 20

    c.save()


def generate_sample_documents(name, dob, address, wealth, funds):
    # Generates four sample PDFs so the UI can simulate uploaded KYC documents.
    # Downstream agents only check upload presence and filenames in this prototype.
    folder = Path(tempfile.mkdtemp())

    files = {
        "government_id": folder / "id.pdf",
        "proof_of_address": folder / "address.pdf",
        "wealth_document": folder / "wealth.pdf",
        "bank_statement": folder / "funds.pdf",
    }

    _create_sample_pdf(files["government_id"], "ID", [name, dob])
    _create_sample_pdf(files["proof_of_address"], "Address", [address])
    _create_sample_pdf(files["wealth_document"], "Wealth", [wealth])
    _create_sample_pdf(files["bank_statement"], "Funds", [funds])

    return tuple(str(p) for p in files.values())


def _document_summary(uploaded_file):
    # Converts an uploaded file path into the flat document structure used by agents.
    # This keeps the case_package readable and avoids nested file objects in output.
    if uploaded_file is None:
        return {"uploaded": False, "filename": None}

    file_path = Path(uploaded_file)
    return {"uploaded": True, "filename": file_path.name}


def generate_sample_case():
    # Generates a deterministic high-risk HNW demo persona.
    # The fixed name is used by simulated screening and business-registry matches.
    name = "Jame Adreen"
    dob = fake.date_of_birth(minimum_age=45, maximum_age=75).isoformat()
    rel = "Individual"
    address = fake.random_element(elements=CANADIAN_SAMPLE_ADDRESSES)
    citizenships = ["Canada"]

    # Contact fields are included in the UI case package for completeness.
    phone = fake.phone_number()
    email = fake.email()

    # Identity document metadata is declared by the client in the prototype.
    doc_type = "Passport"
    issuing_country = "Canada"
    expiry = fake.future_date(end_date="+8y").isoformat()

    # Customer profile values are intentionally high-risk for the demo workflow.
    occupation = "Retired Business Owner"
    income = "1M-5M"
    net_worth = "25M-50M"
    pep = "Yes"

    # Tax profile is simple in this prototype; US status is derived later in the case package.
    tax_res = "Canada"

    # Account usage values trigger enhanced review indicators downstream.
    purpose = "Private Banking"
    activity = "Cross-border transactions"
    cross_border = "Yes"
    third_party = "Yes"

    # Wealth and funds values create a complex source-of-wealth/source-of-funds scenario.
    wealth = "Business sale"
    funds = "Crypto wallet / exchange transfer"
    crypto = "Yes"

    # Business sale context triggers the business structure review path.
    business_sale_context = "Yes"
    consent = "Yes"

    gov_id, addr_doc, wealth_doc, bank_doc = generate_sample_documents(
        name, dob, address, wealth, funds
    )

    # Return values are ordered to match the Gradio output components below.
    return (
        name, dob, rel, address, citizenships,
        phone, email,
        doc_type, issuing_country, expiry,
        occupation, income, net_worth, pep,
        tax_res,
        purpose, activity, cross_border, third_party,
        wealth, funds, crypto,
        business_sale_context, consent,
        gov_id, addr_doc, wealth_doc, bank_doc
    )


def build_case_package(*vals):
    # Converts UI field values into the structured case_package used by the workflow.
    # This is the main object passed to the Orchestrator, agents, Signal Aggregator, and Risk Engine.
    (
        name, dob, rel, address, citizenships,
        phone, email,
        doc_type, issuing_country, expiry,
        occupation, income, net_worth, pep,
        tax_res,
        purpose, activity, cross_border, third_party,
        wealth, funds, crypto,
        business_sale_context, consent,
        gov_id, addr_doc, wealth_doc, bank_doc
    ) = vals

    return {
        # Case ID is generated once here and passed through the full workflow unchanged.
        "case_id": f"KYC-{uuid4().hex[:8].upper()}",
        "created_at": _toronto_now_iso(),

        # CRM context tells downstream logic what type of client segment this is.
        "crm_context": {
            "client_segment": "HNW"
        },

        # Profile stores client-declared onboarding data in clear domain sections.
        "profile": {
            "client_identity": {
                "full_name": name,
                "date_of_birth": dob,
                "relationship_type": rel,
                "residential_address": address,
                "country_of_residence": "Canada",
                "citizenships": citizenships
            },
            "contact": {
                "phone_number": phone,
                "email_address": email
            },
            "identity_documents": {
                "document_type": doc_type,
                "issuing_country": issuing_country,
                "expiry_date": expiry
            },
            "client_profile": {
                "occupation": occupation,
                "estimated_income_range": income,
                "estimated_net_worth_range": net_worth,
                "pep_declaration": pep
            },
            "tax_profile": {
                "primary_tax_residency": tax_res,
                "us_person_status": "Yes" if tax_res == "US" else "No"
            },
            "account_usage": {
                "purpose_of_relationship": purpose,
                "expected_account_activity": activity,
                "cross_border_activity": cross_border,
                "third_party_transactions": third_party
            },
            "wealth_profile": {
                "source_of_wealth": wealth,
                "source_of_funds": funds,
                "crypto_exposure": crypto
            },
            "business_context": {
                "business_sale_context_present": business_sale_context
            },
            "consents": {
                "credit_bureau_consent": consent
            }
        },

        # Flat document references make it easy for agents to check upload presence.
        # The prototype does not parse or verify the contents of these files.
        "documents": {
            "government_id": _document_summary(gov_id),
            "proof_of_address": _document_summary(addr_doc),
            "wealth_document": _document_summary(wealth_doc),
            "bank_statement": _document_summary(bank_doc)
        }
    }


def generate_full_case():
    # Wrapper used by the Gradio button to populate the UI with a sample case.
    return generate_sample_case()


# Gradio UI for generating and inspecting the prototype onboarding case.
with gr.Blocks() as demo:
    gr.Markdown("## HNW KYC Onboarding Prototype")

    generate_btn = gr.Button("Generate Sample Case")

    # Identity and contact inputs.
    full_name = gr.Textbox(label="Name")
    date_of_birth = gr.Textbox(label="DOB")
    relationship_type = gr.Textbox(label="Relationship Type")
    residential_address = gr.Textbox(label="Address")
    citizenships = gr.Textbox(label="Citizenships")

    phone = gr.Textbox(label="Phone")
    email = gr.Textbox(label="Email")

    # Identity document inputs.
    document_type = gr.Textbox(label="Doc Type")
    issuing_country = gr.Textbox(label="Issuing Country")
    expiry_date = gr.Textbox(label="Expiry")

    # Customer profile and tax inputs.
    occupation = gr.Textbox(label="Occupation")
    estimated_income_range = gr.Textbox(label="Income")
    estimated_net_worth_range = gr.Textbox(label="Net Worth")
    pep_declaration = gr.Textbox(label="PEP")

    tax_residency_country = gr.Textbox(label="Tax Residency")

    # Account usage inputs used by screening and wealth/risk logic.
    purpose = gr.Textbox(label="Purpose")
    activity = gr.Textbox(label="Activity")
    cross_border = gr.Textbox(label="Cross-border")
    third_party = gr.Textbox(label="Third-party")

    # Wealth, funds, and business context inputs.
    wealth = gr.Textbox(label="Wealth Source")
    funds = gr.Textbox(label="Funds Source")
    crypto = gr.Textbox(label="Crypto")

    business = gr.Textbox(label="Business Sale Context Present")
    consent = gr.Textbox(label="Consent")

    # Document upload inputs used by downstream agents as evidence references.
    gov = gr.File(label="Government ID")
    addr = gr.File(label="Proof of Address")
    wealth_doc = gr.File(label="Wealth Document")
    bank_doc = gr.File(label="Bank Statement")

    # Populates the UI with the deterministic high-risk sample case.
    generate_btn.click(
        fn=generate_full_case,
        inputs=[],
        outputs=[
            full_name, date_of_birth, relationship_type, residential_address,
            citizenships, phone, email, document_type, issuing_country,
            expiry_date, occupation, estimated_income_range,
            estimated_net_worth_range, pep_declaration, tax_residency_country,
            purpose, activity, cross_border, third_party, wealth, funds, crypto,
            business, consent, gov, addr, wealth_doc, bank_doc,
        ],
    )

demo.launch(share=False, debug=False)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.9 MB/s eta 0:00:00
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [ ]:
#  Create Sample Case Package
# This creates the structured case_package used by the workflow controller.
# In a production system, this object would come from the UI or case service.
sample = generate_sample_case()
case_package = build_case_package(*sample)

# Optional inspection:
#print(json.dumps(case_package, indent=2))

# The Orchestrator

The Orchestrator decides which agents need to run for this case.

It looks at the case package and asks:

**Which parts of this client profile need review?**

For this high-net-worth case, the Orchestrator sees:

- High-net-worth client
- Business sale context
- Crypto exposure
- Cross-border activity
- Third-party transaction expectations

Based on that, it creates a review plan.

The review plan says which agents should run and what information each agent should use.

For this case, the plan includes:

- Identity Verification Agent
- Screening Agent
- Wealth and Funds Review Agent
- Business Structure Review Agent

The Orchestrator does **not** run these agents.

That job belongs to the Workflow Controller.

The Orchestrator also does **not** make the risk decision.

That job belongs to the Risk Engine.

In simple terms:

- The Orchestrator creates the plan
- The Workflow Controller runs the agents
- The specialist agents produce findings
- The Signal Aggregator organizes the findings
- The Risk Engine makes the rules-based risk decision

In [ ]:
# @title
import json

from datetime import datetime
from zoneinfo import ZoneInfo


def _toronto_now_iso():
    # Timestamp orchestrator output in Toronto time for the Canada-focused workflow.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


def _get_nested(source, path, default=None):
    # Safely reads nested dictionary values without failing if an optional section is missing.
    current = source

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

        if current is None:
            return default

    return current


def _is_yes(value):
    # Normalizes Yes/No-style fields so routing rules are consistent.
    return str(value).strip().lower() == "yes"


def run_kyc_orchestrator(case_package):
    # The orchestrator receives the case package and creates a routing plan.
    # It does not run agents and does not make the final risk decision.
    case_id = case_package["case_id"]

    # Read only the fields needed to decide which checks should run.
    profile = _get_nested(case_package, ["profile"], {})
    business_context = _get_nested(profile, ["business_context"], {})

    business_sale_context_present = business_context.get(
        "business_sale_context_present"
    )

    # These checks always run for the HNW onboarding prototype.
    required_checks = [
        "identity_verification",
        "screening",
        "wealth_and_funds_review",
    ]

    # Business sale context triggers the business structure review path.
    if _is_yes(business_sale_context_present):
        required_checks.append("business_structure_review")

    # Dispatch instructions tell each specialist agent which case_package sections to use.
    # This keeps agent inputs explicit, traceable, and easy to audit.
    dispatch_instructions = {
        "identity_verification": {
            "method": "DUAL_PROCESS",
            "use": [
                "profile.client_identity",
                "profile.identity_documents",
                "profile.consents",
                "documents.government_id",
                "documents.proof_of_address"
            ]
        },
        "screening": {
            "use": [
                "profile.client_identity",
                "profile.client_profile",
                "profile.tax_profile"
            ]
        },
        "wealth_and_funds_review": {
            "use": [
                "profile.wealth_profile",
                "profile.account_usage",
                "documents.wealth_document",
                "documents.bank_statement"
            ]
        },
    }

    # Add business review instructions only when that review is required.
    if "business_structure_review" in required_checks:
        dispatch_instructions["business_structure_review"] = {
            "use": [
                "profile.business_context",
                "profile.client_identity",
                "profile.wealth_profile"
            ]
        }

    # Return the plan for the Workflow Controller.
    # The Workflow Controller will execute the required agents.
    return {
        "case_id": case_id,
        "orchestrator_name": "kyc_orchestrator",
        "execution_status": "COMPLETED",
        "created_at": _toronto_now_iso(),
        "required_checks": required_checks,
        "dispatch_instructions": dispatch_instructions,
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# orchestrator_output = run_kyc_orchestrator(case_package)
# print(json.dumps(orchestrator_output, indent=2))


# The Identity Verification Agent

The Identity Verification Agent checks whether the client's identity information and identity documents are ready for verification.

In production, this would use approved identity verification systems, document verification tools, credit bureau checks, or other permitted identity verification methods.

In this prototype, we do not call any external identity verification provider.

Instead, the prototype checks whether the required identity inputs are present:
- Client identity information
- Government ID
- Proof of address
- Required consents



In [ ]:
# @title
import json
from datetime import datetime
from zoneinfo import ZoneInfo


def _toronto_now_iso():
    # Timestamp agent output in Toronto time for the Canada-focused workflow.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


def _get_nested(source, path, default=None):
    # Safely reads nested dictionary values from the case package or orchestrator output.
    current = source

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

        if current is None:
            return default

    return current


def _has_value(value):
    # Checks whether a required onboarding field has been provided.
    if value is None:
        return False

    if isinstance(value, str):
        return bool(value.strip())

    if isinstance(value, dict):
        return bool(value)

    if isinstance(value, list):
        return bool(value)

    return True


def _is_uploaded(document_ref):
    # Checks whether a prototype document reference shows that a file was uploaded.
    # This does not verify the document's authenticity or contents.
    return (
        isinstance(document_ref, dict)
        and document_ref.get("uploaded") is True
    )


def run_identity_verification(case_package: dict, orchestrator_output: dict) -> dict:
    # Read the identity verification instructions created by the orchestrator.
    # The agent follows the plan; it does not decide its own verification method.
    identity_instruction = _get_nested(
        orchestrator_output,
        ["dispatch_instructions", "identity_verification"],
        {},
    )

    inputs_used = identity_instruction.get("use", [])
    method_used = identity_instruction.get("method")

    # Read only the identity-related fields assigned to this agent.
    client_identity = _get_nested(case_package, ["profile", "client_identity"], {})
    identity_documents = _get_nested(case_package, ["profile", "identity_documents"], {})
    credit_bureau_consent = _get_nested(
        case_package,
        ["profile", "consents", "credit_bureau_consent"],
    )
    government_id = _get_nested(case_package, ["documents", "government_id"])
    proof_of_address = _get_nested(case_package, ["documents", "proof_of_address"])

    # Convert required identity inputs into normalized readiness signals.
    client_identity_present = _has_value(client_identity)
    identity_documents_present = _has_value(identity_documents)
    government_id_present = _is_uploaded(government_id)
    proof_of_address_present = _is_uploaded(proof_of_address)
    credit_bureau_consent_present = credit_bureau_consent == "Yes"

    # DUAL_PROCESS readiness requires identity data, supporting documents, and consent.
    # This is only an input-readiness check; no external verification is performed.
    dual_process_ready = (
        method_used != "DUAL_PROCESS"
        or (
            client_identity_present
            and identity_documents_present
            and government_id_present
            and proof_of_address_present
            and credit_bureau_consent_present
        )
    )

    # Return normalized signals for the Signal Aggregator.
    # The finding says evidence is ready, not that identity has been externally verified.
    return {
        "agent_name": "identity_verification_agent",
        "execution_status": "COMPLETED",
        "created_at": _toronto_now_iso(),
        "finding": "IDENTITY_EVIDENCE_READY",
        "inputs_used": inputs_used,
        "signals": {
            "identity_method": method_used,
            "client_identity_present": client_identity_present,
            "identity_documents_present": identity_documents_present,
            "government_id_present": government_id_present,
            "proof_of_address_present": proof_of_address_present,
            "credit_bureau_consent_present": credit_bureau_consent_present,
            "dual_process_ready": dual_process_ready,
            "external_identity_verification_performed": False,
        },
        "summary": (
            "The Identity Verification Agent confirmed that the required identity "
            "onboarding inputs for the DUAL_PROCESS method were present."
        ),
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# identity_result = run_identity_verification(
#     case_package,
#     orchestrator_output,
# )
# print(json.dumps(identity_result, indent=2))


# The Screening Agent

The Screening Agent checks the client against sanctions, PEP, adverse media, and internal watchlists.

In a real bank, this would use approved vendor systems and internal databases for screening.

In this prototype, we use a simulated database to check for PEP matches, sanctions matches, and internal watchlist matches.

In [ ]:
# @title
import json

from datetime import datetime
from zoneinfo import ZoneInfo


def _toronto_now_iso():
    # Timestamp screening output in Toronto time for the Canada-focused workflow.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


# Small local registry used only to demonstrate deterministic screening behavior.
# This is not a real sanctions, PEP, or internal watchlist data source.
SIMULATED_SCREENING_DB = {
    "pep_registry": [
        {
            "full_name": "Jame Adreen",
            "category": "DOMESTIC_PEP",
        },
        {
            "full_name": "Elena Markovic",
            "category": "FOREIGN_PEP",
        },
    ],
    "sanctions_registry": [
        {
            "full_name": "Victor Petrov",
            "source": "SIMULATED_OFAC",
        },
    ],
    "internal_watchlist": [
        {
            "full_name": "Michael Brown",
            "reason": "PREVIOUS_FRAUD_ALERT",
        },
    ],
}


def _exact_name_match(full_name, registry):
    # Performs exact deterministic matching only.
    # No fuzzy matching, scoring, external provider lookup, or AI reasoning is used.
    if not _has_value(full_name):
        return False

    normalized_name = str(full_name).strip()

    for record in registry:
        if record.get("full_name") == normalized_name:
            return True

    return False


def run_screening(case_package: dict, orchestrator_output: dict) -> dict:
    # Read the screening instructions created by the orchestrator.
    # The agent uses these instructions to record which fields were assigned to screening.
    screening_instruction = _get_nested(
        orchestrator_output,
        ["dispatch_instructions", "screening"],
        {},
    )

    inputs_used = screening_instruction.get("use", [])

    # Read only the onboarding fields relevant to this prototype screening check.
    full_name = _get_nested(
        case_package,
        ["profile", "client_identity", "full_name"],
    )

    pep_declaration = _get_nested(
        case_package,
        ["profile", "client_profile", "pep_declaration"],
    )

    us_person_status_value = _get_nested(
        case_package,
        ["profile", "tax_profile", "us_person_status"],
    )

    # Convert declared onboarding values into normalized screening signals.
    client_identity_present = _has_value(full_name)
    pep_declared = _is_yes(pep_declaration)
    us_person_status = _is_yes(us_person_status_value)

    # Simulate local registry screening with exact-name matching.
    pep_match_detected = _exact_name_match(
        full_name,
        SIMULATED_SCREENING_DB["pep_registry"],
    )

    sanctions_match_detected = _exact_name_match(
        full_name,
        SIMULATED_SCREENING_DB["sanctions_registry"],
    )

    internal_watchlist_match_detected = _exact_name_match(
        full_name,
        SIMULATED_SCREENING_DB["internal_watchlist"],
    )

    # Assign a single screening finding using the highest-priority indicator.
    if sanctions_match_detected:
        finding = "SANCTIONS_MATCH_DETECTED"
    elif internal_watchlist_match_detected:
        finding = "INTERNAL_WATCHLIST_MATCH_DETECTED"
    elif pep_match_detected:
        finding = "PEP_MATCH_DETECTED"
    elif pep_declared:
        finding = "PEP_DECLARED"
    else:
        finding = "NO_SCREENING_INDICATORS"

    # Build a short human-readable summary of the simulated screening result.
    summary_parts = []

    if pep_declared and pep_match_detected:
        summary_parts.append(
            "The Screening Agent identified a client-declared PEP status and a simulated PEP registry match for the onboarding profile."
        )
    elif pep_declared:
        summary_parts.append(
            "The Screening Agent identified a client-declared PEP status for the onboarding profile."
        )
    elif pep_match_detected:
        summary_parts.append(
            "The Screening Agent identified a simulated PEP registry match for the onboarding profile."
        )

    if sanctions_match_detected:
        summary_parts.append(
            "The Screening Agent identified a simulated sanctions registry match for the onboarding profile."
        )

    if internal_watchlist_match_detected:
        summary_parts.append(
            "The Screening Agent identified a simulated internal watchlist match for the onboarding profile."
        )

    if summary_parts:
        summary = " ".join(summary_parts)
    else:
        summary = (
            "The Screening Agent completed deterministic onboarding screening checks for the client profile."
        )

    # Return normalized screening signals for the Signal Aggregator.
    # This output demonstrates workflow integration, not real commercial screening.
    return {
        "agent_name": "screening_agent",
        "execution_status": "COMPLETED",
        "created_at": _toronto_now_iso(),
        "finding": finding,
        "inputs_used": inputs_used,
        "signals": {
            "client_identity_present": client_identity_present,
            "pep_declared": pep_declared,
            "us_person_status": us_person_status,
            "pep_match_detected": pep_match_detected,
            "sanctions_match_detected": sanctions_match_detected,
            "internal_watchlist_match_detected": internal_watchlist_match_detected,
        },
        "summary": summary,
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# screening_result = run_screening(
#     case_package,
#     orchestrator_output,
# )
# print(json.dumps(screening_result, indent=2))


# The Wealth and Funds Review Agent

The Wealth and Funds Review Agent reviews the client's source of wealth and source of funds.

**Source of wealth** means: How did the client become wealthy over time?

**Source of funds** means: Where is the specific money entering this banking relationship coming from?

In a real bank, this step may review business sale documents, bank statements, tax documents, and crypto exchange statements. It may also use AI to organize complex documentation and highlight what the compliance officer should review.

In this prototype, the agent checks whether the client has declared source of wealth and source of funds, whether supporting documents are present, and whether risk indicators exist:
- Business sale wealth source
- Crypto exposure
- Cross-border activity
- Third-party transactions


In [ ]:
# @title
import json

from datetime import datetime
from zoneinfo import ZoneInfo


def _toronto_now_iso():
    # Timestamp wealth review output in Toronto time for the Canada-focused workflow.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


def _get_nested(source, path, default=None):
    # Safely reads nested dictionary values from the case package or orchestrator output.
    current = source

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

        if current is None:
            return default

    return current


def _has_value(value):
    # Checks whether a declared wealth/funds field has been provided.
    if value is None:
        return False

    if isinstance(value, str):
        return bool(value.strip())

    if isinstance(value, dict):
        return bool(value)

    if isinstance(value, list):
        return bool(value)

    return True


def _is_uploaded(document_ref):
    # Checks whether a supporting document was uploaded in the prototype.
    # This does not parse, validate, or authenticate the document.
    return (
        isinstance(document_ref, dict)
        and document_ref.get("uploaded") is True
    )


def _is_yes(value):
    # Normalizes Yes/No-style values into booleans for signal generation.
    return str(value).strip().lower() == "yes"


def _contains_text(value, text):
    # Lightweight keyword check used for deterministic prototype indicators.
    return text.lower() in str(value or "").strip().lower()


def run_wealth_and_funds_review(case_package: dict, orchestrator_output: dict) -> dict:
    # Read the wealth review instructions created by the orchestrator.
    # This records which case fields were assigned to the wealth review step.
    wealth_instruction = _get_nested(
        orchestrator_output,
        ["dispatch_instructions", "wealth_and_funds_review"],
        {},
    )

    inputs_used = wealth_instruction.get("use", [])

    # Read declared source of wealth, source of funds, and account activity fields.
    source_of_wealth = _get_nested(
        case_package,
        ["profile", "wealth_profile", "source_of_wealth"],
    )

    source_of_funds = _get_nested(
        case_package,
        ["profile", "wealth_profile", "source_of_funds"],
    )

    crypto_exposure = _get_nested(
        case_package,
        ["profile", "wealth_profile", "crypto_exposure"],
    )

    cross_border_activity = _get_nested(
        case_package,
        ["profile", "account_usage", "cross_border_activity"],
    )

    third_party_transactions = _get_nested(
        case_package,
        ["profile", "account_usage", "third_party_transactions"],
    )

    # Read supporting document references assigned to this review.
    wealth_document = _get_nested(
        case_package,
        ["documents", "wealth_document"],
    )

    bank_statement = _get_nested(
        case_package,
        ["documents", "bank_statement"],
    )

    # Convert required declarations and documents into normalized readiness signals.
    source_of_wealth_present = _has_value(source_of_wealth)
    source_of_funds_present = _has_value(source_of_funds)
    wealth_document_present = _is_uploaded(wealth_document)
    bank_statement_present = _is_uploaded(bank_statement)

    # Identify simple deterministic wealth and funds indicators.
    business_sale_wealth_source = _contains_text(
        source_of_wealth,
        "business sale",
    )

    crypto_source_of_funds = _contains_text(
        source_of_funds,
        "crypto",
    )

    crypto_exposure_declared = _is_yes(crypto_exposure)
    cross_border_activity_declared = _is_yes(cross_border_activity)
    third_party_transactions_declared = _is_yes(third_party_transactions)

    # Enhanced review is triggered by complex or higher-risk wealth/funds indicators.
    enhanced_review_required = (
        business_sale_wealth_source
        or crypto_source_of_funds
        or crypto_exposure_declared
        or cross_border_activity_declared
        or third_party_transactions_declared
    )

    # This prototype checks whether supporting evidence was uploaded.
    # It does not inspect the content of the uploaded files.
    support_evidence_present = (
        source_of_wealth_present
        and source_of_funds_present
        and wealth_document_present
        and bank_statement_present
    )

    # Produce a finding for the Signal Aggregator.
    if support_evidence_present and enhanced_review_required:
        finding = "WEALTH_EVIDENCE_PRESENT_ENHANCED_REVIEW_REQUIRED"
    else:
        finding = "WEALTH_EVIDENCE_PRESENT"

    # Explain the wealth review outcome in plain English for downstream summaries.
    if enhanced_review_required:
        summary = (
            "The Wealth and Funds Review Agent confirmed that declared source of wealth, "
            "source of funds, wealth documentation, and bank statement evidence were present. "
            "The profile includes business sale wealth context, crypto-related source of funds, "
            "cross-border activity, and third-party transaction indicators requiring enhanced "
            "wealth and funds review."
        )
    else:
        summary = (
            "The Wealth and Funds Review Agent confirmed that declared source of wealth, "
            "source of funds, wealth documentation, and bank statement evidence were present."
        )

    # Return normalized signals for the Signal Aggregator.
    # These signals support risk routing but do not approve or reject the case.
    return {
        "agent_name": "wealth_and_funds_review_agent",
        "execution_status": "COMPLETED",
        "created_at": _toronto_now_iso(),
        "finding": finding,
        "inputs_used": inputs_used,
        "signals": {
            "source_of_wealth_present": source_of_wealth_present,
            "source_of_funds_present": source_of_funds_present,
            "wealth_document_present": wealth_document_present,
            "bank_statement_present": bank_statement_present,
            "business_sale_wealth_source": business_sale_wealth_source,
            "crypto_source_of_funds": crypto_source_of_funds,
            "crypto_exposure_declared": crypto_exposure_declared,
            "cross_border_activity_declared": cross_border_activity_declared,
            "third_party_transactions_declared": third_party_transactions_declared,
            "enhanced_review_required": enhanced_review_required,
        },
        "summary": summary,
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# wealth_result = run_wealth_and_funds_review(
#     case_package,
#     orchestrator_output,
# )
# print(json.dumps(wealth_result, indent=2))


# The Business Structure Review Agent

The Business Structure Review Agent checks whether the client was connected to the business that was sold.

In a real bank, this step may use corporate registries, ownership records, and business databases.

In this prototype, we use a simulated business registry. The agent checks whether the client's declared business sale matches a simulated business ownership database.


In [ ]:
# @title
import json
from datetime import datetime
from zoneinfo import ZoneInfo


def _toronto_now_iso():
    # Timestamp business structure output in Toronto time for the Canada-focused workflow.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


# Small local registry used only to simulate business-sale context in the prototype.
# This is not a real corporate registry, ownership registry, or UBO data source.
SIMULATED_BUSINESS_REGISTRY = {
    "Jame Adreen": {
        "business_name": "Adreen Capital Holdings Inc.",
        "ownership_status": "FORMER_OWNER",
        "sale_context": "BUSINESS_SOLD",
        "registry_match_detected": True,
    }
}


def _get_nested(source, path, default=None):
    # Safely reads nested dictionary values from the case package or orchestrator output.
    current = source

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

        if current is None:
            return default

    return current


def _has_value(value):
    # Checks whether a required business context field has been provided.
    if value is None:
        return False

    if isinstance(value, str):
        return bool(value.strip())

    if isinstance(value, dict):
        return bool(value)

    if isinstance(value, list):
        return bool(value)

    return True


def _is_yes(value):
    # Normalizes Yes/No-style values into booleans for signal generation.
    return str(value).strip().lower() == "yes"


def _contains_text(value, text):
    # Lightweight keyword check used for deterministic prototype indicators.
    return text.lower() in str(value or "").strip().lower()


def run_business_structure_review(
    case_package: dict,
    orchestrator_output: dict,
) -> dict:
    # Read the business review instructions created by the orchestrator.
    # This records which case fields were assigned to the business structure review step.
    business_instruction = _get_nested(
        orchestrator_output,
        ["dispatch_instructions", "business_structure_review"],
        {},
    )

    inputs_used = business_instruction.get("use", [])

    # Read the client identity, business context, and wealth fields needed for this check.
    full_name = _get_nested(
        case_package,
        ["profile", "client_identity", "full_name"],
    )

    relationship_type = _get_nested(
        case_package,
        ["profile", "client_identity", "relationship_type"],
    )

    business_sale_context_value = _get_nested(
        case_package,
        ["profile", "business_context", "business_sale_context_present"],
    )

    source_of_wealth = _get_nested(
        case_package,
        ["profile", "wealth_profile", "source_of_wealth"],
    )

    # Perform an exact-name lookup against the simulated business registry.
    registry_record = SIMULATED_BUSINESS_REGISTRY.get(full_name)

    # Convert business context into normalized signals for the Signal Aggregator.
    individual_client = relationship_type == "Individual"

    business_sale_context_present = _is_yes(
        business_sale_context_value
    )

    business_sale_wealth_source = _contains_text(
        source_of_wealth,
        "business sale",
    )

    business_registry_match_detected = (
        isinstance(registry_record, dict)
        and registry_record.get("registry_match_detected") is True
    )

    former_business_owner = (
        isinstance(registry_record, dict)
        and registry_record.get("ownership_status") == "FORMER_OWNER"
    )

    # Confirm business-sale context only when the declaration, wealth source, and registry match align.
    if (
        business_sale_context_present
        and business_sale_wealth_source
        and business_registry_match_detected
    ):
        finding = "BUSINESS_SALE_CONTEXT_CONFIRMED"
    else:
        finding = "NO_BUSINESS_STRUCTURE_INDICATORS"

    # Summarize the business structure result without implying real registry verification.
    if finding == "BUSINESS_SALE_CONTEXT_CONFIRMED":
        summary = (
            "The Business Structure Review Agent identified business sale context "
            "for the onboarding profile and confirmed a simulated former business "
            "ownership match in the local registry."
        )
    else:
        summary = (
            "The Business Structure Review Agent completed deterministic business "
            "structure review checks for the onboarding profile."
        )

    # Return normalized signals for the Signal Aggregator.
    # No UBO review, registry API call, or ownership-chain analysis is performed.
    return {
        "agent_name": "business_structure_review_agent",
        "execution_status": "COMPLETED",
        "created_at": _toronto_now_iso(),
        "finding": finding,
        "inputs_used": inputs_used,
        "signals": {
            "individual_client": individual_client,
            "business_sale_context_present": business_sale_context_present,
            "business_sale_wealth_source": business_sale_wealth_source,
            "business_registry_match_detected": business_registry_match_detected,
            "former_business_owner": former_business_owner,
            "ubo_review_performed": False,
            "registry_verification_performed": False,
        },
        "summary": summary,
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# business_structure_result = run_business_structure_review(
#     case_package,
#     orchestrator_output,
# )
# print(json.dumps(business_structure_result, indent=2))


# The Signal Aggregator

The Signal Aggregator combines the outputs from the specialist agents into one unified view that's easy for the Risk Engine to consume.

For this prototype, each agent produces its own finding:
- Identity Agent → identity evidence ready
- Screening Agent → PEP match detected
- Wealth Agent → enhanced wealth review required
- Business Agent → business sale context confirmed

The Risk Engine should not need to understand every agent's internal format.

The Signal Aggregator converts the separate agent outputs into a consistent structure so the Risk Engine can apply its rules.

It does not make the risk decision. It only prepares the case-level input for the Risk Engine.


In [ ]:
# @title
import json
from datetime import datetime
from zoneinfo import ZoneInfo


def _toronto_now_iso():
    # Timestamp aggregation output in Toronto time for the Canada-focused workflow.
    return datetime.now(ZoneInfo("America/Toronto")).isoformat()


def run_signal_aggregator(
    identity_result: dict,
    screening_result: dict,
    wealth_result: dict,
    business_structure_result: dict,
) -> dict:
    # Capture whether each specialist agent completed successfully.
    # This gives the Risk Engine and reviewers basic execution visibility.
    agent_execution_summary = {
        "identity_verification_agent": identity_result["execution_status"],
        "screening_agent": screening_result["execution_status"],
        "wealth_and_funds_review_agent": wealth_result["execution_status"],
        "business_structure_review_agent": business_structure_result["execution_status"],
    }

    # Preserve each agent's finding without changing it.
    # The aggregator organizes outputs but does not reinterpret agent results.
    agent_findings = {
        "identity_verification_agent": identity_result["finding"],
        "screening_agent": screening_result["finding"],
        "wealth_and_funds_review_agent": wealth_result["finding"],
        "business_structure_review_agent": business_structure_result["finding"],
    }

    # Preserve each agent's plain-English summary for audit and review context.
    agent_summaries = {
        "identity_verification_agent": identity_result["summary"],
        "screening_agent": screening_result["summary"],
        "wealth_and_funds_review_agent": wealth_result["summary"],
        "business_structure_review_agent": business_structure_result["summary"],
    }

    # Track which case_package paths each agent reviewed.
    # This keeps data lineage explicit across the workflow.
    agent_inputs_used = {
        "identity_verification_agent": identity_result["inputs_used"],
        "screening_agent": screening_result["inputs_used"],
        "wealth_and_funds_review_agent": wealth_result["inputs_used"],
        "business_structure_review_agent": business_structure_result["inputs_used"],
    }

    # Group normalized signals by review domain.
    # This creates a clean interface between specialist agents and the Risk Engine.
    aggregated_signals = {
        "identity": identity_result["signals"],
        "screening": screening_result["signals"],
        "wealth_and_funds": wealth_result["signals"],
        "business_structure": business_structure_result["signals"],
    }

    # Convert screening outputs into case-level risk indicators.
    pep_risk_present = (
        screening_result["signals"]["pep_declared"] is True
        or screening_result["signals"]["pep_match_detected"] is True
    )

    sanctions_risk_present = (
        screening_result["signals"]["sanctions_match_detected"] is True
    )

    internal_watchlist_risk_present = (
        screening_result["signals"]["internal_watchlist_match_detected"] is True
    )

    # Convert wealth and business outputs into case-level indicators.
    wealth_enhanced_review_required = (
        wealth_result["signals"]["enhanced_review_required"] is True
    )

    business_sale_context_confirmed = (
        business_structure_result["finding"] == "BUSINESS_SALE_CONTEXT_CONFIRMED"
    )

    # This is only a case-level indicator, not a final decision.
    # The Risk Engine uses these indicators to assign the review route.
    human_review_recommended = (
        pep_risk_present
        or sanctions_risk_present
        or internal_watchlist_risk_present
        or wealth_enhanced_review_required
        or business_sale_context_confirmed
    )

    # Package the indicators into a compact structure for deterministic risk rules.
    case_indicators = {
        "identity_ready": identity_result["finding"] == "IDENTITY_EVIDENCE_READY",
        "pep_risk_present": pep_risk_present,
        "sanctions_risk_present": sanctions_risk_present,
        "internal_watchlist_risk_present": internal_watchlist_risk_present,
        "wealth_enhanced_review_required": wealth_enhanced_review_required,
        "business_sale_context_confirmed": business_sale_context_confirmed,
        "human_review_recommended": human_review_recommended,
    }

    summary = (
        "The Signal Aggregator consolidated specialist agent outputs, preserved agent "
        "findings and summaries, and produced normalized case-level indicators for the Risk Engine."
    )

    # Return the normalized aggregation layer.
    # The Risk Engine consumes this output to apply deterministic review-routing rules.
    return {
        "aggregator_name": "signal_aggregator",
        "execution_status": "COMPLETED",
        "created_at": _toronto_now_iso(),
        "agent_execution_summary": agent_execution_summary,
        "agent_findings": agent_findings,
        "agent_inputs_used": agent_inputs_used,
        "aggregated_signals": aggregated_signals,
        "case_indicators": case_indicators,
        "agent_summaries": agent_summaries,
        "summary": summary,
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# signal_aggregator_output = run_signal_aggregator(
#     identity_result,
#     screening_result,
#     wealth_result,
#     business_structure_result,
# )
# print(json.dumps(signal_aggregator_output, indent=2))


# The Risk Engine and AI Case Summary

The Risk Engine applies rules to the inputs from the Signal Aggregator.

It decides how the case should be handled based on signals from the specialist agents: identity verification, screening, wealth review, and business review.

**In a real bank**, the Risk Engine would apply the bank's internal risk rules to route cases correctly and assign the appropriate risk tier.

**In this prototype**, the Risk Engine uses simplified rules to demonstrate the decision logic.

For this case, the Risk Engine assigns a **HIGH** risk tier and routes the client to **enhanced due diligence review**.

It makes this decision because the workflow identified:
- PEP exposure
- Business sale wealth context
- Crypto exposure
- Cross-border activity
- Third-party transaction expectations

**The Risk Engine does not use an AI model to make the risk decision.**

That is intentional. The bank needs to explain which rules were applied and which evidence triggered human review.

## AI Case Summary

After the Risk Engine applies its rules, an AI agent converts the structured output into plain English.

**In a real bank**, this summary would be based on the actual Risk Engine output and would help the compliance officer understand the case quickly.

**In this prototype**, the AI summary uses Gemini to turn the Risk Engine output into a clear case summary.

The AI explains the risk decision, but it does not make the risk decision, approve the client, or assign the risk tier.

It explains:
- Why the case received that risk rating
- Which areas need human review
- Where the compliance officer should focus their attention

In [ ]:
# @title

!pip install -q google-genai

import json
import os
from google.colab import userdata


# Load the Gemini API key from Colab Secrets.
# Gemini is optional: if the key is missing or the call fails, the Risk Engine uses a rule-based summary.
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")


def _gemini_available() -> bool:
    # Checks whether a Gemini API key is available in the notebook environment.
    return bool(os.environ.get("GEMINI_API_KEY"))


def _generate_template_reviewer_summary(summary_input: dict) -> str:
    # Generates a deterministic fallback summary when Gemini is unavailable.
    # This keeps the Risk Engine usable without any external model call.
    review_route = summary_input["review_route"]

    if review_route == "SANCTIONS_ESCALATION":
        return (
            "This case requires sanctions escalation because a sanctions screening match "
            "was detected. No onboarding decision should be made until sanctions compliance "
            "review is complete."
        )

    if review_route == "FINANCIAL_CRIME_REVIEW":
        return (
            "This case requires financial crime review because an internal watchlist match "
            "was detected. The onboarding case should be reviewed before approval."
        )

    if review_route == "ENHANCED_DUE_DILIGENCE_REVIEW":
        return (
            "This case is classified as HIGH risk and requires enhanced due diligence review. "
            "Key risk factors include confirmed PEP exposure, business sale wealth context, "
            "crypto-related source of funds, expected cross-border activity, and expected "
            "third-party transactions. Identity evidence is present, but the onboarding case "
            "should be reviewed before approval."
        )

    if review_route == "PEP_REVIEW":
        return (
            "This case requires PEP review because PEP exposure is present based on client "
            "declaration or screening match. The onboarding case should be reviewed before approval."
        )

    if review_route == "WEALTH_AND_FUNDS_REVIEW":
        return (
            "This case requires wealth and funds review because enhanced wealth or funds "
            "indicators were identified. The onboarding case should be reviewed before approval."
        )

    return (
        "This case does not contain elevated prototype risk indicators and can proceed "
        "through standard review."
    )


def _generate_llm_reviewer_summary(summary_input: dict) -> str | None:
    # Uses Gemini only to rewrite the deterministic risk decision into reviewer-friendly language.
    # Gemini does not receive raw agent outputs, raw documents, or the full aggregated signal set.
    if not _gemini_available():
        return None

    try:
        from google import genai

        client = genai.Client()

        # The prompt restricts Gemini to summarization only.
        # The deterministic risk tier, review route, and recommended action are already set.
        prompt = f"""
You are writing a plain-English reviewer summary from a deterministic KYC risk decision.

Rules:
- Use only the provided JSON.
- Only restate the provided deterministic risk decision.
- Do not add facts.
- Do not infer new risks.
- Do not infer anything beyond the provided JSON.
- Do not add regulatory explanations.
- Do not explain why each risk driver is risky.
- Do not change the risk tier.
- Do not change the review route.
- Do not change the recommended action.
- Do not approve or reject the client.
- Do not add phrases like "can complicate traceability", "jurisdictional complexity", or "obscure beneficial ownership".
- Use plain English.
- Use natural enterprise compliance-review language.
- Avoid awkward phrasing such as "risk present from".
- Prefer wording like "risk identified through", "risk confirmed through", or "risk detected through".
- When describing enhanced wealth and funds review, integrate it naturally into the sentence instead of phrasing it as a separate requirement.
- Write one concise paragraph.
- Do not use markdown, bullets, bold text, or numbered lists.
- Use a professional compliance-review tone.

Preferred style example:
This case is classified as HIGH risk due to PEP risk identified through both client declaration and screening registry match. The risk profile also includes business sale wealth context, crypto exposure in the source of funds or wealth profile, expected cross-border account activity, expected third-party transactions, and a requirement for enhanced wealth and funds review. Route the case to enhanced due diligence review before onboarding approval.

Deterministic risk decision JSON:
{json.dumps(summary_input, indent=2)}
"""

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )

        return response.text.strip()

    except Exception as e:
        # Keep the workflow resilient if Gemini is unavailable, rate-limited, or misconfigured.
        print("Gemini summary generation failed:", e)
        return None


def run_risk_engine(
    signal_aggregator_output: dict,
    use_llm_summary: bool = True,
) -> dict:
    # Read the normalized aggregation output.
    # The Risk Engine consumes signals; it does not inspect raw documents or run agents.
    case_indicators = signal_aggregator_output["case_indicators"]
    aggregated_signals = signal_aggregator_output["aggregated_signals"]
    agent_findings = signal_aggregator_output["agent_findings"]
    information_reviewed = signal_aggregator_output["agent_inputs_used"]

    # Pull case-level indicators used by the deterministic routing rules.
    pep_risk_present = case_indicators["pep_risk_present"]
    sanctions_risk_present = case_indicators["sanctions_risk_present"]
    internal_watchlist_risk_present = case_indicators["internal_watchlist_risk_present"]
    wealth_enhanced_review_required = case_indicators["wealth_enhanced_review_required"]
    business_sale_context_confirmed = case_indicators["business_sale_context_confirmed"]

    # Read detailed signals needed to explain risk drivers.
    wealth_signals = aggregated_signals["wealth_and_funds"]
    screening_signals = aggregated_signals["screening"]

    crypto_source_of_funds = wealth_signals["crypto_source_of_funds"]
    crypto_exposure_declared = wealth_signals["crypto_exposure_declared"]
    cross_border_activity_declared = wealth_signals["cross_border_activity_declared"]
    third_party_transactions_declared = wealth_signals["third_party_transactions_declared"]
    business_sale_wealth_source = wealth_signals["business_sale_wealth_source"]

    pep_declared = screening_signals["pep_declared"]
    pep_match_detected = screening_signals["pep_match_detected"]

    # Deterministic risk decision logic.
    # Gemini never assigns or changes risk tier, review route, or recommended action.
    if sanctions_risk_present:
        risk_tier = "PROHIBITED_OR_ESCALATE"
        review_route = "SANCTIONS_ESCALATION"
        human_review_required = True

    elif internal_watchlist_risk_present:
        risk_tier = "HIGH"
        review_route = "FINANCIAL_CRIME_REVIEW"
        human_review_required = True

    elif pep_risk_present and wealth_enhanced_review_required:
        risk_tier = "HIGH"
        review_route = "ENHANCED_DUE_DILIGENCE_REVIEW"
        human_review_required = True

    elif pep_risk_present:
        risk_tier = "MEDIUM_HIGH"
        review_route = "PEP_REVIEW"
        human_review_required = True

    elif wealth_enhanced_review_required:
        risk_tier = "MEDIUM"
        review_route = "WEALTH_AND_FUNDS_REVIEW"
        human_review_required = True

    else:
        risk_tier = "LOW"
        review_route = "STANDARD_REVIEW"
        human_review_required = False

    # Build plain-English risk drivers from the deterministic signals.
    # These drivers explain why the review route was selected.
    risk_drivers = []

    if pep_declared and pep_match_detected:
        risk_drivers.append(
            "PEP risk was identified through both client declaration and screening registry match."
        )
    elif pep_declared:
        risk_drivers.append("PEP risk was identified through client declaration.")
    elif pep_match_detected:
        risk_drivers.append("PEP risk was identified through screening registry match.")

    if sanctions_risk_present:
        risk_drivers.append("A sanctions screening match was detected.")

    if internal_watchlist_risk_present:
        risk_drivers.append("An internal watchlist match was detected.")

    if wealth_enhanced_review_required:
        risk_drivers.append("Enhanced wealth and funds review is required.")

    if business_sale_wealth_source and business_sale_context_confirmed:
        risk_drivers.append(
            "Business sale wealth context was declared and confirmed by the business structure review."
        )
    elif business_sale_wealth_source:
        risk_drivers.append("The declared source of wealth is a business sale.")

    if crypto_source_of_funds or crypto_exposure_declared:
        risk_drivers.append(
            "Crypto exposure is present in the source of funds or wealth profile."
        )

    if cross_border_activity_declared:
        risk_drivers.append("Cross-border account activity is expected.")

    if third_party_transactions_declared:
        risk_drivers.append("Third-party transactions are expected.")

    # Recommended action follows directly from the deterministic review route.
    recommended_actions = {
        "SANCTIONS_ESCALATION": (
            "Escalate the case to sanctions compliance review before any onboarding decision."
        ),
        "FINANCIAL_CRIME_REVIEW": (
            "Route the case to financial crime review before onboarding approval."
        ),
        "ENHANCED_DUE_DILIGENCE_REVIEW": (
            "Route the case to enhanced due diligence review before onboarding approval."
        ),
        "PEP_REVIEW": (
            "Route the case to PEP review before onboarding approval."
        ),
        "WEALTH_AND_FUNDS_REVIEW": (
            "Route the case to wealth and funds review before onboarding approval."
        ),
        "STANDARD_REVIEW": (
            "Proceed with standard onboarding review."
        ),
    }

    recommended_action = recommended_actions[review_route]

    # This is the only package sent to Gemini.
    # It contains the final deterministic decision, not raw agent outputs.
    summary_input = {
        "risk_tier": risk_tier,
        "review_route": review_route,
        "human_review_required": human_review_required,
        "risk_drivers": risk_drivers,
        "recommended_action": recommended_action,
    }

    rule_based_summary = _generate_template_reviewer_summary(summary_input)

    if use_llm_summary:
        llm_summary = _generate_llm_reviewer_summary(summary_input)
    else:
        llm_summary = None

    # Use Gemini summary when available; otherwise fall back to deterministic template text.
    reviewer_summary = llm_summary if llm_summary else rule_based_summary

    # Return the final risk output for the Workflow Controller.
    # This output does not approve or reject the client.
    return {
        "risk_engine_name": "kyc_risk_engine",
        "execution_status": "COMPLETED",
        "decision_method": "DETERMINISTIC_RULES",
        "risk_tier": risk_tier,
        "review_route": review_route,
        "human_review_required": human_review_required,
        "information_reviewed": information_reviewed,
        "agent_findings": agent_findings,
        "risk_drivers": risk_drivers,
        "recommended_action": recommended_action,
        "reviewer_summary": reviewer_summary,
    }


# Optional debug run. Keep commented so the Workflow Controller remains the single execution entry point.
# risk_engine_output = run_risk_engine(
#     signal_aggregator_output,
#     use_llm_summary=True,
# )
# print(json.dumps(risk_engine_output, indent=2))


# The Workflow Controller

The Workflow Controller coordinates the full process:
1. Receives the case package
2. Calls the Orchestrator to get the review plan
3. Runs the required specialist agents
4. Collects the agent outputs
5. Sends the outputs to the Signal Aggregator
6. Sends the aggregated case view to the Risk Engine
7. Returns the final workflow output

When you run the Workflow Controller, it returns one JSON output that shows:
- Which agents ran
- Which inputs each agent reviewed
- The final risk decision
- The recommended next action
- The case summary for the compliance officer

In [ ]:
# @title
# This is the single entry point for running the full KYC prototype.
# All previous cells should define functions only.

import json


def run_kyc_workflow(
    case_package: dict,
    use_llm_summary: bool = True,
    include_debug_trace: bool = False,
) -> dict:
    # Start the workflow by asking the orchestrator for a routing plan.
    # The controller follows this plan instead of deciding required checks itself.
    orchestrator_output = run_kyc_orchestrator(case_package)

    required_checks = orchestrator_output["required_checks"]

    # Store agent outputs by check name so they can be passed to the aggregator.
    agent_outputs = {}

    # Run identity verification only if the orchestrator included it in the plan.
    if "identity_verification" in required_checks:
        identity_result = run_identity_verification(
            case_package,
            orchestrator_output,
        )
        agent_outputs["identity_verification"] = identity_result

    # Run screening only if required by the orchestrator.
    if "screening" in required_checks:
        screening_result = run_screening(
            case_package,
            orchestrator_output,
        )
        agent_outputs["screening"] = screening_result

    # Run wealth and funds review only if required by the orchestrator.
    if "wealth_and_funds_review" in required_checks:
        wealth_result = run_wealth_and_funds_review(
            case_package,
            orchestrator_output,
        )
        agent_outputs["wealth_and_funds_review"] = wealth_result

    # Run business structure review only when business context triggers that check.
    if "business_structure_review" in required_checks:
        business_structure_result = run_business_structure_review(
            case_package,
            orchestrator_output,
        )
        agent_outputs["business_structure_review"] = business_structure_result

    # This prototype persona is expected to run all four agents.
    # Pull each result out explicitly for the current aggregator interface.
    identity_result = agent_outputs["identity_verification"]
    screening_result = agent_outputs["screening"]
    wealth_result = agent_outputs["wealth_and_funds_review"]
    business_structure_result = agent_outputs["business_structure_review"]

    # Consolidate specialist outputs into normalized signals and case-level indicators.
    signal_aggregator_output = run_signal_aggregator(
        identity_result,
        screening_result,
        wealth_result,
        business_structure_result,
    )

    # Send aggregated signals to the Risk Engine for deterministic routing.
    # Gemini, if enabled, is used only for the reviewer summary.
    risk_engine_output = run_risk_engine(
        signal_aggregator_output,
        use_llm_summary=use_llm_summary,
    )

    # Keep the clean output focused on completion status by agent.
    agent_execution_summary = signal_aggregator_output[
        "agent_execution_summary"
    ]

    # Use the Risk Engine's reviewed input paths instead of hardcoded labels.
    inputs_reviewed = risk_engine_output["information_reviewed"]

    # Present the final risk outcome in a compact workflow summary.
    workflow_summary = {
        "risk_tier": risk_engine_output["risk_tier"],
        "review_route": risk_engine_output["review_route"],
        "human_review_required": risk_engine_output["human_review_required"],
        "recommended_action": risk_engine_output["recommended_action"],
        "reviewer_summary": risk_engine_output["reviewer_summary"],
    }

    # Return a clean high-level workflow output for normal notebook use.
    workflow_output = {
        "workflow_name": "kyc_workflow_controller",
        "execution_status": "COMPLETED",
        "case_id": case_package["case_id"],
        "agent_execution_summary": agent_execution_summary,
        "inputs_reviewed": inputs_reviewed,
        "workflow_summary": workflow_summary,
    }

    # Optional debug trace exposes the full internal execution path for demos or troubleshooting.
    if include_debug_trace:
        workflow_output["debug_trace"] = {
            "required_checks": required_checks,
            "orchestrator_output": orchestrator_output,
            "agent_outputs": agent_outputs,
            "signal_aggregator_output": signal_aggregator_output,
            "risk_engine_output": risk_engine_output,
        }

    return workflow_output


# Run the full prototype end-to-end from the controller.
kyc_workflow_output = run_kyc_workflow(
    case_package,
    use_llm_summary=True,
    include_debug_trace=False,
)

print(json.dumps(kyc_workflow_output, indent=2))

# Print the reviewer summary separately so it is easier to read than the JSON string view.
#print("\nReviewer Summary:\n")
#print(kyc_workflow_output["workflow_summary"]["reviewer_summary"])


{
  "workflow_name": "kyc_workflow_controller",
  "execution_status": "COMPLETED",
  "case_id": "KYC-B8A581B1",
  "agent_execution_summary": {
    "identity_verification_agent": "COMPLETED",
    "screening_agent": "COMPLETED",
    "wealth_and_funds_review_agent": "COMPLETED",
    "business_structure_review_agent": "COMPLETED"
  },
  "inputs_reviewed": {
    "identity_verification_agent": [
      "profile.client_identity",
      "profile.identity_documents",
      "profile.consents",
      "documents.government_id",
      "documents.proof_of_address"
    ],
    "screening_agent": [
      "profile.client_identity",
      "profile.client_profile",
      "profile.tax_profile"
    ],
    "wealth_and_funds_review_agent": [
      "profile.wealth_profile",
      "profile.account_usage",
      "documents.wealth_document",
      "documents.bank_statement"
    ],
    "business_structure_review_agent": [
      "profile.business_context",
      "profile.client_identity",
      "profile.wealth_profi